# Dieses Notebook dient der Konsolidierung einzelner DataFrames aus den Datenquellen

Weiterhin wird jedes der 3 finalen DataFrames (df_solldaten_cis, df_istdaten_emr, df_istdaten_prc) so normalisiert und vorbereitet, dass alle Mafi Touren entfernt werden, Kennzeichen sowie Fahrer normalisiert werden, die Rohdatenquellen auf die relevanten Inhalte reduziert werden, die Zeitstempel entsprechend geparst werden und die Daten letztlich auf Tour-oder Stopp-Ebene vorliegen.

## 1. Struktur der PRC-Dateien sichten

Die PRC-Dateien sind heterogenes XML: eine einzelne Datei kann mehrere Meldungen unterschiedlichen Typs enthalten. Welche Typen es gibt, steht in der C-Logistic-Schnittstellenbeschreibung: Position, FMSStatus, Tourstatus, Stationsstatus und Sendposstatus. Bevor die Daten eingelesen werden, geben wir der Übersicht halber je ein vollständiges Beispiel pro Typ aus, damit wir sehen, wie die Meldungen aufgebaut sind und welche Felder wir später übernehmen.

In [1]:
import re
import pandas as pd
from pathlib import Path

# Anzeigeoptionen, damit DataFrames vollständig sichtbar sind
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", None)

BASE_PATH = Path.cwd().parent / "data" / "raw"

# Alle PRC-Dateien finden
prc_files = sorted(BASE_PATH.glob("msg_*.prc"))
print(f"Gefundene PRC-Dateien: {len(prc_files)}")

# # Die fünf Meldungstypen laut C-Logistic-Schnittstellenbeschreibung
bekannte_typen = ["Sendposstatus", "Stationsstatus", "FMSStatus", "Tourstatus", "Position"]

# Pro Typ die erste Datei finden, in der er vorkommt
beispiele = {}
for pfad in prc_files:
    with open(pfad, "r", encoding="utf-8") as f:
        inhalt = f.read()
    for typ in bekannte_typen:
        if typ not in beispiele and f"<{typ}>" in inhalt:
            beispiele[typ] = (pfad.name, inhalt)
    if len(beispiele) == len(bekannte_typen):
        break

print(f"Beispiele für die {len(bekannte_typen)} Meldungstypen:")

# Beispielhafte Inhalte der gefundenen Meldungstypen anzeigen
for typ, (name, inhalt) in beispiele.items():
    print(f"Meldungstyp: {typ}    (Datei: {name})")
    print(inhalt)

Gefundene PRC-Dateien: 24521
Beispiele für die 5 Meldungstypen:
Meldungstyp: Position    (Datei: msg_20260228000043_956.imp.20260228000310889.prc)
﻿<Import>
  <Position>
    <GeoPosition>
      <PositionsID>16049334470</PositionsID>
      <Longitude>9.98548</Longitude>
      <Latitude>53.52335</Latitude>
      <Zeitstempel_GPS>2026-02-27T23:59:59+01:00</Zeitstempel_GPS>
      <Richtung>0</Richtung>
      <Geschwindigkeit>0</Geschwindigkeit>
      <KMStand>339687</KMStand>
    </GeoPosition>
    <Zeitstempel_Fahrzeug>2026-02-28T00:00:00+01:00</Zeitstempel_Fahrzeug>
    <Zeitstempel_Server>2026-02-28T00:00:43</Zeitstempel_Server>
    <FahrzeugID>Plö-TS856</FahrzeugID>
  </Position>
</Import>
Meldungstyp: FMSStatus    (Datei: msg_20260301141707_983.imp.20260301141842821.prc)
﻿<Import>
  <Tourstatus>
    <GeoPosition>
      <PositionsID>16053897222</PositionsID>
      <Longitude>9.98927</Longitude>
      <Latitude>53.52262</Latitude>
      <Zeitstempel_GPS>2026-03-01T14:13:25+01:00</Zeitst

Bevor wir einlesen, halten wir fest, wie sich die Typen unterscheiden und welchen Mehrwert sie für das weitere Vorgehen haben könnten, aus den Beispielen oben und aus der C-Logistic-Beschreibung:

- **Position** nur GPS (Ort, Geschwindigkeit, Kilometerstand) mit Zeitstempel und Fahrzeug, kein Status, kein Tour- oder Stationsbezug. Die lückenlose Positionsspur und damit potenzielle Grundlage für Geofencing, Route und Standzeit.
- **Stationsstatus** trägt eine StationID und einen Status und bildet den Stations-Zyklus einer Tour ab, von der Anfahrt bis zur Abfahrt. Dürfte die zentrale Quelle für Standzeiten und die tatsächliche Stopp-Reihenfolge sein.
- **Sendposstatus** trägt eine SendposID und einen Status und bezieht sich auf eine einzelne Sendungsposition. Könnte zeigen, was an einem Stopp tatsächlich geladen oder entladen wurde.
- **Tourstatus** trägt eine TourID und einen Status auf Tour-Ebene und klammert eine Tour zeitlich.
- **FMSStatus** scheinen Fahrzeugsignale der Flotten-Management-Schnittstelle zu sein, in Werten, ohne Status und ohne Tour- oder Stationsbezug. 

## 2. Einlesen und Zusammenführen der PRC-Meldungen

Jede Datei hat ein < Import > Wurzelelement, dessen direkte Kinder die einzelnen Meldungen sind (eine Datei kann mehrere enthalten). Wir erzeugen daher eine Zeile je Kind und übernehmen den Tag-Namen als msg_typ.

Pro Meldung behalten wir nur die Felder, die wir für den Soll-Ist-Vergleich und die Routen-/Standzeitanalyse brauchen:

- aus der GeoPosition: Längen-/Breitengrad, Geschwindigkeit, Kilometerstand und der Zeitstempel_GPS als Grundlage für Route und Standzeit. Richtung lassen wir weg, da sich die Fahrtrichtung aus der Abfolge der Positionen ergibt.
- auf Meldungsebene: Fahrzeug, Zeitstempel_Fahrzeug und Zeitstempel_Server, welcher sich als Meldungszeitpunkt besser eignet, klären wir noch, der Status sowie die Schlüssel TourID, StationID und SendposID. Die Schlüssel übernehmen wir unverändert und vollständig.

Den Werte-Block lesen wir bei den Statusmeldungen nur bei FMSStatus mit ein, deren Bedeutung müsste nochmal mit Frank geklärt oder als Muster identifiziert werden. Bei Tour- und
Stationsstatus ist er in den Beispieldaten leer. Lediglich bei Sendposstatus trägt er Inhalte über Liefermengen (Gewicht, Anzahl Paletten). Die brauchen wir nicht, da unser
Vergleich den Ablauf misst und in den Soll-Daten die Informationen zu den Liefermengen im Tourverlauf immer gleich bleibt.

In [2]:
import xml.etree.ElementTree as ET

# Felder unter <GeoPosition>
GEO_FELDER = ["PositionsID", "Longitude", "Latitude", "Geschwindigkeit", "KMStand", "Zeitstempel_GPS"]
# Direkte Felder unter dem Meldungselement
MELDUNG_FELDER = ["Zeitstempel_Fahrzeug", "Zeitstempel_Server", "FahrzeugID",
                  "Status", "TourID", "StationID", "SendposID"]

def parse_prc_datei(pfad):
    """Eine Zeile je Meldungselement unter <Import>. FMSStatus trägt zusätzlich
    mehrere <Werte> (Typ/Wert), die ebenfalls mit gesammelt werden."""
    zeilen, fms_werte = [], []
    root = ET.parse(pfad).getroot()       
    for meldung in root:                  
        eintrag = {"msg_typ": meldung.tag, "quelle_datei": pfad.name}
        geo = meldung.find("GeoPosition")
        for feld in GEO_FELDER:
            eintrag[feld] = geo.findtext(feld)
        for feld in MELDUNG_FELDER:
            eintrag[feld] = meldung.findtext(feld)
        zeilen.append(eintrag)
        if meldung.tag == "FMSStatus":
            for w in meldung.findall("Werte"):
                fms_werte.append({"quelle_datei": pfad.name,
                                  "PositionsID": geo.findtext("PositionsID"),
                                  "Typ": w.findtext("Typ"), "Wert": w.findtext("Wert")})
    return zeilen, fms_werte

# Alle Dateien parsen und die Ergebnisse sammeln
alle_zeilen, alle_fms = [], []
for pfad in prc_files:
    zeilen, fms = parse_prc_datei(pfad)
    alle_zeilen.extend(zeilen)
    alle_fms.extend(fms)

# DataFrames erstellen und Überblick über die Daten geben
prc_raw = pd.DataFrame(alle_zeilen)
fms_werte_raw = pd.DataFrame(alle_fms)

print(f"Verarbeitete Dateien: {len(prc_files)}")
print(f"Erzeugte Meldungszeilen: {len(prc_raw)}")
print(f"FMS-Werte-Einträge: {len(fms_werte_raw)}")
print("Verteilung der Meldungstypen:")
print(prc_raw["msg_typ"].value_counts(dropna=False))
prc_raw.head()

Verarbeitete Dateien: 24521
Erzeugte Meldungszeilen: 33155
FMS-Werte-Einträge: 43170
Verteilung der Meldungstypen:
msg_typ
Position          11535
FMSStatus          8634
Stationsstatus     8299
Sendposstatus      3280
Tourstatus         1405
Ereignis              2
Name: count, dtype: int64


,msg_typ,quelle_datei,PositionsID,Longitude,Latitude,Geschwindigkeit,KMStand,Zeitstempel_GPS,Zeitstempel_Fahrzeug,Zeitstempel_Server,FahrzeugID,Status,TourID,StationID,SendposID
0,Position,msg_20260228000043_956.imp.20260228000310889.prc,16049334470,9.98548,53.52335,0,339687,2026-02-27T23:59:59+01:00,2026-02-28T00:00:00+01:00,2026-02-28T00:00:43,Plö-TS856,NaN,NaN,NaN,NaN
1,Position,msg_20260228000043_957.imp.20260228000310923.prc,16049339872,9.98613,53.5234,0,502579,2026-02-27T23:59:59+01:00,2026-02-28T00:00:00+01:00,2026-02-28T00:00:43,TS859,NaN,NaN,NaN,NaN
2,Position,msg_20260228000315_958.imp.20260228000815933.prc,16049349544,9.98619,53.52356,0,183989,2026-02-27T20:53:12+01:00,2026-02-28T00:02:42+01:00,2026-02-28T00:03:15,OD-TS 8050,NaN,NaN,NaN,NaN
3,Position,msg_20260228000345_959.imp.20260228000815958.prc,16049351179,9.98592,53.52345,0,461200,2026-02-27T23:59:59+01:00,2026-02-28T00:00:00+01:00,2026-02-28T00:03:45,OD-TS 8046,NaN,NaN,NaN,NaN
4,Position,msg_20260228060024_960.imp.20260228060348749.prc,16050228689,9.98616,53.52342,0,502579,2026-02-28T05:59:59+01:00,2026-02-28T06:00:00+01:00,2026-02-28T06:00:24,TS859,NaN,NaN,NaN,NaN


### 2.1 Der sechste Meldungstyp Ereignis

Die Verteilung zeigt die fünf aus der Dokumentation bekannten Meldungstypen und überraschenderweise einen sechsten undzwar Ereignis, mit nur zwei Vorkommen im ganzen Monat. Das flache Parsing-Schema hat dafür nur die gemeinsamen Felder erfasst, daher lesen wir die rohe XML-Datei direkt aus.

In [3]:
ereignis = prc_raw[prc_raw["msg_typ"] == "Ereignis"]
display(ereignis)

# Alle Dateien anzeigen, die <Ereignis> enthalten
for name in ereignis["quelle_datei"].unique():
    print(name)
    print((BASE_PATH / name).read_text(encoding="utf-8-sig"))

,msg_typ,quelle_datei,PositionsID,Longitude,Latitude,Geschwindigkeit,KMStand,Zeitstempel_GPS,Zeitstempel_Fahrzeug,Zeitstempel_Server,FahrzeugID,Status,TourID,StationID,SendposID
4629,Ereignis,msg_20260305081857_4412.imp.20260305082219283.prc,16083444083,9.96948,53.51279,0,461908,2026-03-05T08:15:03+01:00,2026-03-05T08:15:03+01:00,2026-03-05T08:18:57,OD-TS 8046,NaN,NaN,NaN,NaN
21442,Ereignis,msg_20260319132430_16778.imp.20260319132701458.prc,16183750851,9.92304,53.59549,0,807024,2026-03-19T13:24:28+01:00,2026-03-19T13:24:28+01:00,2026-03-19T13:24:30,WL-PL431,NaN,NaN,NaN,NaN


msg_20260305081857_4412.imp.20260305082219283.prc
<Import>
  <FMSStatus>
    <GeoPosition>
      <PositionsID>16083444083</PositionsID>
      <Longitude>9.96948</Longitude>
      <Latitude>53.51279</Latitude>
      <Zeitstempel_GPS>2026-03-05T08:15:03+01:00</Zeitstempel_GPS>
      <Richtung>0</Richtung>
      <Geschwindigkeit>0</Geschwindigkeit>
      <KMStand>461908</KMStand>
    </GeoPosition>
    <Zeitstempel_Fahrzeug>2026-03-05T08:15:03+01:00</Zeitstempel_Fahrzeug>
    <Zeitstempel_Server>2026-03-05T08:18:57</Zeitstempel_Server>
    <FahrzeugID>OD-TS 8046</FahrzeugID>
    <FMSStatusID>16083444083</FMSStatusID>
    <Werte>
      <Typ>4</Typ>
      <Wert>0</Wert>
    </Werte>
    <Werte>
      <Typ>1</Typ>
      <Wert>0</Wert>
    </Werte>
    <Werte>
      <Typ>5</Typ>
      <Wert>0</Wert>
    </Werte>
    <Werte>
      <Typ>2</Typ>
      <Wert>461908.531</Wert>
    </Werte>
    <Werte>
      <Typ>4</Typ>
      <Wert>0</Wert>
    </Werte>
  </FMSStatus>
  <Ereignis>
    <GeoPosition

Im rohen XML trägt jede Ereignis-Meldung ein eigenes Feld Typ, welches bei beiden Fällen 3 ist und eine EreignisID, dazu GeoPosition, Zeitstempel und Fahrzeug. Allerdings keinen Status und
keinen Tour-, Stations- oder Sendpos-Schlüssel. Laut Mapping-XML entspricht Typ 3 dem Ereignis „Beginn-Stau". Es sind also fahrzeugbezogene Ereignismeldungen ohne Bindung an eine Station oder Tour, hier jeweils zusammen mit einer FMSStatus-Meldung in derselben Datei. Damit wäre erklärt, warum bei diesen Zeilen Status und alle Schlüssel leer sind.

Für die Standzeit- und Routenanalyse sind sie ohne Stations- oder Tour-Bezug nicht verwertbar und mit nur zwei Vorkommen im ganzen Monat wirkt das etwas wenig. Wir entfernen diese Zeilen aber nicht, da sie für die Erklärung von Verzögerungen relevant sein könnten. Da sie keinen Status tragen, bekommen sie auch keinen Status-Text und spielen im Vergleich entsprechend keine Rolle. Die
ereignisspezifischen Felder Typ und EreignisID wurden durch das Schema bewusst nicht erfasst, da sie ohnehin entfallen.

### 2.2 Die FMS-Werte sichten

Wir untersuchen die FMS-Signale, die wir separat in fms_werte_raw gesammelt haben und schauen, welche Typ-Codes vorkommen und wie ihre Werte verteilt sind.

In [4]:
fms_werte_raw["Wert"] = pd.to_numeric(fms_werte_raw["Wert"], errors="coerce")

print("Häufigkeit je Typ-Code:")
print(fms_werte_raw["Typ"].value_counts())
print("\nWerteverteilung je Typ-Code:")
print(fms_werte_raw.groupby("Typ")["Wert"].describe())

Häufigkeit je Typ-Code:
Typ
4    17268
1     8634
5     8634
2     8634
Name: count, dtype: int64

Werteverteilung je Typ-Code:
       count           mean            std       min         25%       50%  \
Typ                                                                          
1     8634.0      18.209266      28.507100       0.0       0.000       0.0   
2     8634.0  344070.385756  197234.933548  143450.3  184216.313  315364.3   
4    17268.0    8705.683896   19185.153761       0.0       0.000       0.0   
5     8634.0  -32872.099259   65946.988797 -154875.0       0.000       0.0   

          75%         max  
Typ                        
1        38.8      96.800  
2    502675.1  808361.063  
4         0.0   53598.000  
5         0.0   43990.000  


describe() zeigt, dass Typ 2 als einziges immer belegt ist (min 143.450, nie 0) und im selben Wertebereich liegt wie KMStand (bis 808.361). Typ 1 liegt zwischen 0 und ~97 mit Median 0.
Typ 4 zwischen 0 und ~53.600, mit Median 0, kommt außerdem doppelt vor. Typ 5 kann negativ sein (−154.875 bis 43.990) mit Median 0.

Typ 2 liegt in den XML-Dateien immer sehr nah am Kilometerstand, der zu dieser Zeit zurückgemeldet wurde. Das wird nochmal vertieft geprüft, sobald die Daten preprocessed sind.

Ansonsten müssten diese Typ Codes nochmal mit Frank oder Kai Wallaschek besprochen werden.

### 2.3 Technische Erstprüfung der Meldungstypen

In den Rohdaten fällt auf, dass die Zeitstempel unterschiedlich aufgebaut sind (Zeitstempel_GPS -und _Fahrzeug mit Zeitzonen-Offset, _Server ohne Offset). Bevor wir bereinigen, verschaffen wir uns mit .info() einen Überblick über Spalten, Datentypen und Non-Null.

In [5]:
prc_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 33155 entries, 0 to 33154
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   msg_typ               33155 non-null  str  
 1   quelle_datei          33155 non-null  str  
 2   PositionsID           33155 non-null  str  
 3   Longitude             33155 non-null  str  
 4   Latitude              33155 non-null  str  
 5   Geschwindigkeit       33155 non-null  str  
 6   KMStand               33155 non-null  str  
 7   Zeitstempel_GPS       33155 non-null  str  
 8   Zeitstempel_Fahrzeug  33155 non-null  str  
 9   Zeitstempel_Server    33155 non-null  str  
 10  FahrzeugID            33155 non-null  str  
 11  Status                12984 non-null  str  
 12  TourID                1405 non-null   str  
 13  StationID             8299 non-null   str  
 14  SendposID             3280 non-null   str  
dtypes: str(15)
memory usage: 9.4 MB


## 3. Bereinigung und Vorbereitung
### 3.1 Zeitstempel parsen
Anhand von .info folgt der erste Arbeitsschritt, der eigentlich auf der Hand lag. Dadurch, dass wir die Dateien als XML eingelesen haben, liegen alle Spalten String vor und werden in die passenden Typen überführt. Wir legen dafür in prc eine Arbeitskopie an, prc_raw bleibt unverändert.

Wir parsen Zeitstempel_GPS und Zeitstempel_Fahrzeug über UTC und rechnen sie auf lokale Zeit (Europe/Berlin) um, anschließend ohne Zeitzone, damit alle drei als lokale, tz-lose Zeit vorliegen und mit den ebenfalls tz-losen Zeiten aus Soll und EMR besser vergleichbar sind. Die Sommerzeit-Umstellung Ende März wird dabei automatisch korrekt behandelt.

Bei den numerischen Feldern werden Längen-/Breitengrad, Geschwindigkeit und Kilometerstand zu Zahlen. Status und Ereignis_Typ werden als nullable Integer geführt, da sie nur bei ihren jeweiligen Meldungstypen belegt sind. Die ID-Felder (PositionsID, TourID, StationID, SendposID) bleiben bewusst Text.

Alle Umwandlungen laufen mit errors="coerce", sodass nicht-parsebare Werte als Leerwert sichtbar werden. 

In [6]:
prc = prc_raw.copy()

# Zeitstempel mit Offset über UTC auf lokale, tz-lose Zeit (Europe/Berlin) bringen
for col in ["Zeitstempel_GPS", "Zeitstempel_Fahrzeug"]:
    prc[col] = pd.to_datetime(prc[col], utc=True).dt.tz_convert("Europe/Berlin").dt.tz_localize(None)
prc["Zeitstempel_Server"] = pd.to_datetime(prc["Zeitstempel_Server"])

# GPS-Felder numerisch, Status als nullable Integer
for col in ["Longitude", "Latitude", "Geschwindigkeit", "KMStand"]:
    prc[col] = pd.to_numeric(prc[col])
prc["Status"] = pd.to_numeric(prc["Status"]).astype("Int64")

prc.dtypes

msg_typ                            str
quelle_datei                       str
PositionsID                        str
Longitude                      float64
Latitude                       float64
Geschwindigkeit                  int64
KMStand                          int64
Zeitstempel_GPS         datetime64[us]
Zeitstempel_Fahrzeug    datetime64[us]
Zeitstempel_Server      datetime64[us]
FahrzeugID                         str
Status                           Int64
TourID                             str
StationID                          str
SendposID                          str
dtype: object

Nachdem wir die Zeitstempel geparsed haben, prüfen wir die drei Zeitbereiche der Zeitstempel_GPS/Fahrzeug/Server. Daraus ergibt sich, ob und wie wir auf den gemeinsamen Analysezeitraum für März filtern müssen. Außerdem entscheiden wir so, welcher Zeitstempel als fachlicher Meldungszeitpunkt für den Filter dient.

In [7]:
print("Zeitbereich je Zeitstempel-Spalte:")
for col in ["Zeitstempel_GPS", "Zeitstempel_Fahrzeug", "Zeitstempel_Server"]:
    print(f"  {col}: {prc[col].min()}  bis  {prc[col].max()}")

print("\nMeldungen je Kalendertag (Zeitstempel_Fahrzeug):")
print(prc["Zeitstempel_Fahrzeug"].dt.date.value_counts().sort_index())

Zeitbereich je Zeitstempel-Spalte:
  Zeitstempel_GPS: 2026-02-09 12:19:59  bis  2026-03-31 13:25:19
  Zeitstempel_Fahrzeug: 2026-02-28 00:00:00  bis  2026-03-31 13:25:22
  Zeitstempel_Server: 2026-02-28 00:00:43  bis  2026-03-31 13:26:09

Meldungen je Kalendertag (Zeitstempel_Fahrzeug):
Zeitstempel_Fahrzeug
2026-02-28      18
2026-03-01      24
2026-03-02    1452
2026-03-03    1232
2026-03-04    1340
2026-03-05    1498
2026-03-06    1150
2026-03-07      24
2026-03-08      24
2026-03-09    1446
2026-03-10    1772
2026-03-11    1930
2026-03-12    1722
2026-03-13    1388
2026-03-14      33
2026-03-15      21
2026-03-16    1590
2026-03-17    1778
2026-03-18    1476
2026-03-19    1851
2026-03-20    1184
2026-03-21      53
2026-03-22      27
2026-03-23    1343
2026-03-24    1243
2026-03-25    1548
2026-03-26    1846
2026-03-27     860
2026-03-28      38
2026-03-29      30
2026-03-30    1482
2026-03-31    1732
Name: count, dtype: int64


### 3.2 Bezugszeitpunkt und Filter auf den Analysezeitraum

Der Zeitbereich zeigt, dass die Zeitstempel unterschiedlich weit zurückreichen. Zeitstempel_GPS beginnt schon am 09.02., während Zeitstempel_Fahrzeug und Zeitstempel_Server erst am 28.02 anfangen. Das zeigt, dass einzelne GPS-Zeitstempel veraltet sein können und sich nicht als Meldungszeitpunkt eignen. Wir verwenden Zeitstempel_Fahrzeug als Referenz-Meldungszeitpunkt, das ist der Zeitpunkt, zu dem das Fahrzeug die Meldung erzeugt hat. Zeitstempel_Server markiert eher den Eingang im Backend.

Zweitens beginnen die Daten Ende Februar. Wie bei Soll und EMR begrenzen wir auf den gemeinsamen Analysezeitraum März und entfernen die 18 Meldungen vom 28.02.

In der Tagesverteilung enthalten einzelne Tage nur rund 20–50 Meldungen statt der etwa 1.200–1.900 an Werktagen, könnte an Mafi liegen, wird noch bereinigt.

In [8]:
# Filter auf den gemeinsamen Analysezeitraum März, gleiche Grenzen wie Soll und EMR
vorher = len(prc)
prc = prc[(prc["Zeitstempel_Fahrzeug"] >= "2026-03-01") &
          (prc["Zeitstempel_Fahrzeug"] <  "2026-04-01")].copy()

print(f"Außerhalb März entfernt: {vorher - len(prc)} Zeilen | verbleibend: {len(prc)}")

Außerhalb März entfernt: 18 Zeilen | verbleibend: 33137


### 3.3 Fahrzeugkennungen
In der Vorschau der Daten konnte man schon sehen, dass auch in den PRC-Daten die Fahrzeugkennungen nicht einheitlich vorliegen. Daher sehen wir uns alle vorkommenden Kennungen an, um die
Schreibvarianten vollständig zu erfassen.

In [9]:
print("Eindeutige Fahrzeugkennungen:", prc["FahrzeugID"].nunique())
prc["FahrzeugID"].value_counts(dropna=False)

Eindeutige Fahrzeugkennungen: 13


FahrzeugID
PI-EN 444     4390
TS859         3794
Plö-TS853     3113
OD-TS 8041    3044
OD-TS 8048    2965
OD-TS 8050    2855
Plö-TS856     2647
WL-PL431      2579
OD-TS 8046    2351
Mafi          1665
OD-TS 8044    1638
PI-EN 900     1267
Plo-TS857      829
Name: count, dtype: int64

Die Übersicht zeigt 13 Kennungen und bestätigt die uneinheitlichen Schreibweisen aus der Vorschau. Gleichzeitig bestätigt sich hier, dass auch Mafi mit 1.665 Meldungen in den PRC-Daten steckt. Dasselbe Fahrzeug haben wir in Soll und EMR auf Vorgabe von Frank entfernt und entfernen es hier ebenso.

Die übrigen Kennungen vereinheitlichen wir mit derselben Funktion wie in den anderen Quellen. Großschreibung, Umlaute aufgelöst, Buchstaben- und Zifferngruppen mit Bindestrich getrennt.
Eine Kennung braucht zusätzlich einen Alias: TS859 entspricht keinem normalen Kennzeichen, gehört aber laut Fahrzeugstammdaten zum Fahrzeug Plö-TS859.
Abschließend benennen wir die Spalte in LKW_KENNZ um, passend zum gemeinsamen Schlüssel von Soll und EMR.

In [10]:
def normalisiere_kennzeichen(wert):
    """Vereinheitlicht Fahrzeugkennungen wie in Soll und EMR: Großschreibung,
    Umlaute aufgelöst, Buchstaben-/Zifferngruppen mit Bindestrich getrennt."""
    if pd.isna(wert):
        return pd.NA
    wert = str(wert).upper().strip()
    wert = wert.replace("Ä", "A").replace("Ö", "O").replace("Ü", "U").replace("ß", "SS")
    return "-".join(re.findall(r"[A-Z]+|[0-9]+", wert))

# Mafi entfernen 
vorher = len(prc)
prc = prc[prc["FahrzeugID"] != "Mafi"].copy()
print(f"Mafi-Zeilen entfernt: {vorher - len(prc)}")

# Normalisieren + Alias TS859 -> PLO-TS-859 
prc["FahrzeugID"] = prc["FahrzeugID"].apply(normalisiere_kennzeichen).replace({"TS-859": "PLO-TS-859"})
prc = prc.rename(columns={"FahrzeugID": "LKW_KENNZ"})

print(f"Zeilen jetzt: {len(prc)}")
print("Eindeutige LKW_KENNZ:", prc["LKW_KENNZ"].nunique())
print(prc["LKW_KENNZ"].value_counts().sort_index())

Mafi-Zeilen entfernt: 1665
Zeilen jetzt: 31472
Eindeutige LKW_KENNZ: 12
LKW_KENNZ
OD-TS-8041    3044
OD-TS-8044    1638
OD-TS-8046    2351
OD-TS-8048    2965
OD-TS-8050    2855
PI-EN-444     4390
PI-EN-900     1267
PLO-TS-853    3113
PLO-TS-856    2647
PLO-TS-857     829
PLO-TS-859    3794
WL-PL-431     2579
Name: count, dtype: int64


## 4. Vertiefte Datenprüfung

Vor dem Export sehen wir uns die in der Übersicht notierten Auffälligkeiten wie mögliche Dubletten, den Kilometerstand und die Plausibilität der Geodaten genauer an.

### 4.1 Dubletten

PositionsID ist kein Zeilen-Schlüssel, da die in einer Datei gebündelten Meldungen (z. B. Sendposstatus und FMSStatus) sich denselben GPS-Eintrag teilen und damit dieselbe PositionsID. Zweitens trägt jede Zeile in quelle_datei ihren Herkunftsnamen. Würden wir den mitvergleichen, wäre keine Zeile je doppelt, weil jede aus einer eigenen Datei stammt. Hinzu kommt, dass dieselbe vom Fahrzeug erzeugte Meldung über mehrere aufeinanderfolgende Importdateien ankommen kann und dabei einen um Sekunden versetzten Server-Eingang tragen, obwohl sie inhaltlich identisch ist. Zu beachten ist zudem, dass PositionsID kein Zeilen-Schlüssel ist, da die in einer Datei gebündelten Meldungen (z. B. Sendposstatus und FMSStatus) sich dieselben GPS-Punkte teilen und damit dieselbe PositionsID. Eine echte Dublette ist daher eine über alle Inhaltsfelder identische Zeile. Zuerst sehen wir uns an, welche Meldungstypen betroffen sind, bevor wir bereinigen.

In [11]:
inhalt_cols = [c for c in prc.columns if c not in ["quelle_datei", "Zeitstempel_Server"]]

dubletten = (prc[prc.duplicated(subset=inhalt_cols, keep=False)]
             .sort_values(["PositionsID", "msg_typ", "quelle_datei"]))

print("Betroffene Zeilen insgesamt:", len(dubletten))
print("Verteilung nach Meldungstyp:")
print(dubletten["msg_typ"].value_counts(dropna=False))
display(dubletten[["msg_typ", "quelle_datei", "PositionsID", "Zeitstempel_Fahrzeug",
                   "Zeitstempel_Server", "Status", "StationID"]].head(12))

Betroffene Zeilen insgesamt: 3156
Verteilung nach Meldungstyp:
msg_typ
FMSStatus         2986
Stationsstatus     170
Name: count, dtype: int64


,msg_typ,quelle_datei,PositionsID,Zeitstempel_Fahrzeug,Zeitstempel_Server,Status,StationID
143,FMSStatus,msg_20260302054940_1063.imp.20260302055135386.prc,16056058618,2026-03-02 05:49:36,2026-03-02 05:49:40,<NA>,NaN
145,FMSStatus,msg_20260302054940_1064.imp.20260302055135419.prc,16056058618,2026-03-02 05:49:36,2026-03-02 05:49:40,<NA>,NaN
157,FMSStatus,msg_20260302055042_1070.imp.20260302055135882.prc,16056063396,2026-03-02 05:50:17,2026-03-02 05:50:42,<NA>,NaN
159,FMSStatus,msg_20260302055042_1071.imp.20260302055135966.prc,16056063396,2026-03-02 05:50:17,2026-03-02 05:50:42,<NA>,NaN
161,FMSStatus,msg_20260302055042_1072.imp.20260302055135999.prc,16056063396,2026-03-02 05:50:17,2026-03-02 05:50:42,<NA>,NaN
182,FMSStatus,msg_20260302060251_1087.imp.20260302060645331.prc,16056162705,2026-03-02 05:59:12,2026-03-02 06:02:51,<NA>,NaN
184,FMSStatus,msg_20260302060251_1088.imp.20260302060645445.prc,16056162705,2026-03-02 05:59:12,2026-03-02 06:02:51,<NA>,NaN
186,FMSStatus,msg_20260302060251_1089.imp.20260302060645467.prc,16056162705,2026-03-02 05:59:12,2026-03-02 06:02:51,<NA>,NaN
205,FMSStatus,msg_20260302060626_1101.imp.20260302060646144.prc,16056184183,2026-03-02 06:05:48,2026-03-02 06:06:26,<NA>,NaN
207,FMSStatus,msg_20260302060626_1102.imp.20260302060646214.prc,16056184183,2026-03-02 06:05:48,2026-03-02 06:06:26,<NA>,NaN


Die betroffenen Zeilen verteilen sich auf die zwei Meldungstypen FMSStatus mit 2.986 und Stationsstatus mit 170. An den Quelldateinamen ist erkennbar, dass dieselbe Meldung in mehreren
aufeinanderfolgenden Dateien desselben Import-Vorgangs ankam (fortlaufende Nummern und Importzeit im Millisekundenabstand). Manche Meldungen kamen sogar zwei- bis viermal. Beim Entfernen fallen
daher weniger Zeilen weg, als hier betroffen sind, weil wir je Meldung eine behalten.

Besonders relevant sind die 170 Stationsstatus, da ein doppelt erfasstes Stationsereignis (z. B. eine Ankunft) später eine Standzeit doppelt zählen würde.

In [12]:
vorher = len(prc)
prc = prc.drop_duplicates(subset=inhalt_cols).copy()

print(f"Zeilen vorher:           {vorher}")
print(f"Inhaltsgleiche entfernt: {vorher - len(prc)}")
print(f"Zeilen jetzt:            {len(prc)}")

Zeilen vorher:           31472
Inhaltsgleiche entfernt: 2043
Zeilen jetzt:            29429


### 4.2 Übersicht: fehlende Werte, eindeutige Werte und Statistik

Da wir zu Beginn nur .info() ausgeführt haben, ergänzen wir hier die Übersicht analog zu Soll und EMR. Also Datentypen, Anzahl und Anteil fehlender Werte sowie die Zahl eindeutiger Werte und dazu die Statistik der numerischen Felder wie dem Kilometerstand und den Geodaten.

In [13]:
prc_overview = pd.DataFrame({
    "datentyp": prc.dtypes.astype(str),
    "fehlende_werte": prc.isna().sum(),
    "fehlende_werte_%": (prc.isna().mean() * 100).round(2),
    "eindeutige_werte": prc.nunique(dropna=True),
})
display(prc_overview)

print("Statistik der numerischen Felder")
display(prc[["Longitude", "Latitude", "Geschwindigkeit", "KMStand"]].describe())

,datentyp,fehlende_werte,fehlende_werte_%,eindeutige_werte
msg_typ,str,0,0.00,6
quelle_datei,str,0,0.00,22753
PositionsID,str,0,0.00,19831
Longitude,float64,0,0.00,8462
Latitude,float64,0,0.00,7635
Geschwindigkeit,int64,0,0.00,118
KMStand,int64,0,0.00,5868
Zeitstempel_GPS,datetime64[us],0,0.00,16459
Zeitstempel_Fahrzeug,datetime64[us],0,0.00,16108
Zeitstempel_Server,datetime64[us],0,0.00,9876


Statistik der numerischen Felder


,Longitude,Latitude,Geschwindigkeit,KMStand
count,29429.000000,29429.000000,29429.000000,29429.000000
mean,9.959395,53.567366,13.517653,284365.169153
std,0.344909,0.190485,22.629284,223999.814903
min,8.165830,51.298860,0.000000,0.000000
25%,9.956840,53.522630,0.000000,179093.000000
50%,9.985700,53.523530,0.000000,195984.000000
75%,9.992030,53.627730,23.000000,462479.000000
max,13.647510,54.781390,128.000000,808364.000000


Die fehlenden Werte folgen durchgängig der typgebundenen Belegung. Status, TourID, StationID und SendposID sind nur bei den Meldungstypen gefüllt, die sie tragen, und entsprechend hoch ist der Fehlanteil über alle Zeilen. Die immer belegten Felder (GPS, Zeitstempel, Fahrzeug) haben keine Lücken. Die Statistik zeigt allerdings einen Kilometerstand mit min = 0 bei einem Median um 196.000 und wir überprüfen außerdem die Spannweite der Koordinaten.

### 4.3 Kilometerstand

Wir sehen uns an, bei welchen Meldungstypen der Kilometerstand = 0 auftritt. Daraus ergibt sich, ob es ein durchgehendes Merkmal einzelner Typen ist oder quer durch die Daten läuft.

In [14]:
n_null = (prc["KMStand"] == 0).sum()
print(f"KMStand == 0: {n_null}  ({n_null / len(prc) * 100:.1f}%)")
print("0-Werte nach Meldungstyp:")
print(prc[prc["KMStand"] == 0]["msg_typ"].value_counts())
print("KMStand je Meldungstyp (min / median / max):")
print(prc.groupby("msg_typ")["KMStand"].agg(["min", "median", "max"]))

KMStand == 0: 5533  (18.8%)
0-Werte nach Meldungstyp:
msg_typ
Position          2565
Stationsstatus    1755
Sendposstatus     1002
Tourstatus         211
Name: count, dtype: int64
KMStand je Meldungstyp (min / median / max):
                   min    median     max
msg_typ                                 
Ereignis        461908  634466.0  807024
FMSStatus       143450  315344.0  808361
Position             0  196324.0  808364
Sendposstatus        0  194994.0  808360
Stationsstatus       0  186750.0  808361
Tourstatus           0  194994.0  807996


Es zeigt sich, dass die 0-Werte nicht an einem einzelnen Meldungstyp hängen. Sie treten bei Position, Stationsstatus, Sendposstatus und Tourstatus auf, während dieselben Typen sonst plausible Kilometerstände tragen (Median rund 187.000–196.000). Nur FMSStatus und Ereignis sind durchgehend echt. Die 0er-Kilometerstände sind also ein quer durch die Typen laufender Teilbestand. Wir schauen uns an, ob sie sich auf bestimmte Fahrzeuge konzentrieren.

In [15]:
print("KMStand=0 Anteil je Fahrzeug (%):")
print(prc.groupby("LKW_KENNZ")["KMStand"]
        .apply(lambda s: (s == 0).mean() * 100).round(1).sort_values(ascending=False))

KMStand=0 Anteil je Fahrzeug (%):
LKW_KENNZ
OD-TS-8044    100.0
PLO-TS-853    100.0
PLO-TS-857    100.0
OD-TS-8041      0.0
OD-TS-8048      0.0
OD-TS-8046      0.0
PI-EN-444       0.0
OD-TS-8050      0.0
PI-EN-900       0.0
PLO-TS-856      0.0
PLO-TS-859      0.0
WL-PL-431       0.0
Name: KMStand, dtype: float64


Die 0-Werte sind tatsächlich fahrzeugbedingt. Konkret sind es drei Fahrzeuge (OD-TS-8044, PLO-TS-853, PLO-TS-857), die durchgehend den Kilometerstand als 0 melden. Es ist also keine situative Lücke, sondern ein Telematik-Merkmal dieser drei Fahrzeuge, deren Kilometerstand nicht übertragen wird. Das könnte etwas mit den Fahrtenschreiber-Fahrzeugen zutun haben, die Frank erwähnt hat.

GPS-Spur, Zeiten und Status dieser Fahrzeuge sind vollständig, nur der Kilometerstand fehlt, daher behalten wir alle Zeilen. Für die spätere Kilometer-Abweichung heißt das, dass für diese drei Fahrzeuge aus PRC kein Kilometerstand vorliegt.

Allerdings hat Typ 2 aus FMSStatus ja durchgängige Werte und lässt im Abgleich mit entsprechenden xml Dateien mit hoher Genauigkeit dem Kilometerstand zuordnen.

In [16]:
print("FMSStatus-Meldungen je Fahrzeug:")
print(prc.loc[prc["msg_typ"] == "FMSStatus", "LKW_KENNZ"].value_counts().sort_index())

FMSStatus-Meldungen je Fahrzeug:
LKW_KENNZ
OD-TS-8041     687
OD-TS-8046     465
OD-TS-8048     665
OD-TS-8050     694
PI-EN-444     1477
PI-EN-900      327
PLO-TS-856     549
PLO-TS-859    1095
WL-PL-431      717
Name: count, dtype: int64


Damit wäre zumindest für die FMSStatus-Meldungen je Fahrzeug bestätigt, dass nur neun Fahrzeuge mit echten Kilometerständen FMSStatus senden. Genau die drei 0-km-Fahrzeuge (OD-TS-8044, PLO-TS-853, PLO-TS-857) tauchen gar nicht auf. Beide Merkmale, kein Kilometerstand und kein FMSStatus, treffen also dieselben drei Fahrzeuge. Das festigt die Annahme, dass diese Daten die Fahrtenschreiber sein müssen. Für die KMStand-Lücke bringt FMS damit nichts, aber verwerfen wollen wir den FMS-Block trotzdem noch nicht. Die Code-Bedeutungen lassen sich womöglich noch klären. Wir exportieren fms_werte daher separat, auf dieselbe Basis wie prc gebracht (März, ohne Mafi, normalisierte Kennzeichen).

In [17]:
# Nur die FMSStatus-Meldungen, die nach der Dubletten-Bereinigung
# übrig sind, identifiziert über ihre PositionsID. März-Filter und
# Mafi-Entfernung werden automatisch geerbt.
fms_meldungen = prc.loc[prc["msg_typ"] == "FMSStatus", ["PositionsID", "quelle_datei", "LKW_KENNZ"]]

fms_werte = fms_werte_raw.merge(fms_meldungen, on=["PositionsID", "quelle_datei"], how="inner")
fms_werte["Wert"] = pd.to_numeric(fms_werte["Wert"])
fms_werte = fms_werte.drop(columns="quelle_datei")

print("FMS-Werte:", len(fms_werte))
print("Zugeordnete Meldungen:", fms_werte["PositionsID"].nunique(), "von", len(fms_meldungen), "FMSStatus")
print(fms_werte["Typ"].value_counts().sort_index())

FMS-Werte: 33380
Zugeordnete Meldungen: 6676 von 6676 FMSStatus
Typ
1     6676
2     6676
4    13352
5     6676
Name: count, dtype: int64


### 4.4 Geodaten
Die Statistik zeigt Längengrad 8,17–13,65 und Breitengrad 51,30–54,78, alles innerhalb Deutschlands und ohne 0-Koordinaten. Zur Bestätigung sehen wir uns die Extrempunkte an und prüfen, ob die weit vom Großraum Hamburg entfernten Meldungen echte Fahrten sind, also viele Meldungen über mehrere Fahrzeuge  oder vereinzelte Ausreißer.

In [18]:
print("Extrempunkte (min/max Längen- und Breitengrad):")
extrem = pd.concat([
    prc.nsmallest(1, "Longitude"), prc.nlargest(1, "Longitude"),
    prc.nsmallest(1, "Latitude"),  prc.nlargest(1, "Latitude"),
])
display(extrem[["msg_typ", "LKW_KENNZ", "Zeitstempel_Fahrzeug",
                "Longitude", "Latitude", "Geschwindigkeit"]])

# Meldungen außerhalb des Großraums Hamburg: echte Fahrten oder Einzelausreißer?
fern = prc[(prc["Longitude"] > 11) | (prc["Longitude"] < 9) |
           (prc["Latitude"] < 52.5) | (prc["Latitude"] > 54)]
print(f"Meldungen außerhalb des Großraums Hamburg: {len(fern)}  ({len(fern) / len(prc) * 100:.1f}%)")
print("davon je Fahrzeug:")
print(fern["LKW_KENNZ"].value_counts())

Extrempunkte (min/max Längen- und Breitengrad):


,msg_typ,LKW_KENNZ,Zeitstempel_Fahrzeug,Longitude,Latitude,Geschwindigkeit
2192,Position,OD-TS-8050,2026-03-03 10:26:18,8.16583,53.28381,11
16650,Stationsstatus,OD-TS-8050,2026-03-16 19:27:11,13.64751,54.14740,0
32928,Position,WL-PL-431,2026-03-31 11:50:06,11.99148,51.29886,26
23632,Stationsstatus,PLO-TS-856,2026-03-23 09:06:43,8.83709,54.78139,0


Meldungen außerhalb des Großraums Hamburg: 1162  (3.9%)
davon je Fahrzeug:
LKW_KENNZ
OD-TS-8048    279
PLO-TS-856    264
OD-TS-8050    257
PLO-TS-853    173
WL-PL-431      55
OD-TS-8041     50
OD-TS-8044     50
PLO-TS-857     34
Name: count, dtype: int64


Die Extrempunkte sind reale Fahrzeuge zu plausiblen Zeiten an stimmigen Orten. Die 1.162 Meldungen außerhalb des Großraums Hamburg (3,9 %) verteilen sich über acht Fahrzeuge mit jeweils dutzenden bis hunderten Meldungen, also echte Langstreckentouren, keine Einzelausreißer. Die Geodaten sind damit plausibel und wir nehmen keine weiteren Änderung vor.

## 5. Export

Die bereinigten PRC-Meldungen (prc) und die separat aufbereiteten FMS-Werte (fms_werte) schreiben wir als CSV und Pickle nach data/processed/. CSV dient der Sichtbarkeit, Pickle
bewahrt die Datentypen (besonders die Zeitstempel und den nullable-Integer-Status), sodass im darauf aufbauenden Folge-Notebook ohne erneute Typumwandlung weitergearbeitet werden kann.

In [21]:
export = Path("..") / "data" / "processed"
export.mkdir(parents=True, exist_ok=True)

for name, df in [("istdaten_prc_clean", prc), ("istdaten_prc_fmswerte", fms_werte)]:
    df.to_csv(export / f"{name}.csv", index=False)
    df.to_pickle(export / f"{name}.pkl")
    print(f"{name}: {len(df)} Zeilen | CSV + Pickle")

istdaten_prc_clean: 29429 Zeilen | CSV + Pickle
istdaten_prc_fmswerte: 33380 Zeilen | CSV + Pickle
